# Genesis — Layer 2 calibration notebook

Sprint_3 (T-C-003) executable notebook that drives the full LHS → BO → CALIBRATION_REPORT pipeline. Run top-to-bottom to reproduce `reports/calibration/<run_id>/`.

Reference: `PRD.md §14` + `TECHNICAL_PLAN.md §4.2`.


In [ ]:
%load_ext autoreload
%autoreload 2

import time
from pathlib import Path

from sim.analysis import write_full_report
from sim.params import ParamSpace
from sim.sweeper import Sweeper

## 1. Configure the sweep

`N_LHS = 256` and `BO_TRIALS = 64` match the T-C-003 brief acceptance criterion. Drop to small numbers for quick iteration; the report renderer is shape-stable across scales.

In [ ]:
SEED = 0
N_LHS = 256
BO_TRIALS = 64
LIFETIMES_PER_ARCHETYPE = 3
FINAL_LIFETIMES_PER_ARCHETYPE = 30
MAX_TICKS = 2000
CI_HALF_WIDTH_MAX_DAYS = 1.0
OUTPUT_ROOT = Path('reports/calibration/')

## 2. Run the calibration

In [ ]:
sweeper = Sweeper(
    base_params=ParamSpace(),
    lifetimes_per_archetype=LIFETIMES_PER_ARCHETYPE,
    max_ticks=MAX_TICKS,
)
t0 = time.time()
run = sweeper.calibrate(
    n_lhs=N_LHS,
    bo_trials=BO_TRIALS,
    seed=SEED,
    final_lifetimes_per_archetype=FINAL_LIFETIMES_PER_ARCHETYPE,
    ci_half_width_max_days=CI_HALF_WIDTH_MAX_DAYS,
)
elapsed = time.time() - t0
print(f'Calibration ran in {elapsed:.1f}s')
print(f'Winning source: {run.winning_source}')
print(f'Objectives passed: {run.final_verdict.passed_count}/{run.final_verdict.total_count}')

## 3. Emit every artifact

`write_full_report` produces `selected_params.json`, `objectives_passed.json`, `archetype_breakdown.json`, `sensitivity_analysis.json`, `bo_trace.json`, `lifetimes.jsonl`, three PNGs, and `CALIBRATION_REPORT.md`.

In [ ]:
run_dir = OUTPUT_ROOT / f'calib_{int(time.time())}_{SEED}'
artifacts = write_full_report(run=run, out_dir=run_dir)
for name, path in sorted(artifacts.items()):
    print(f'  - {name}: {path}')

## 4. Per-objective audit

Surface the GOOD_CALIBRATION verdict inline so the notebook is self-documenting.

In [ ]:
for o in run.final_verdict.objectives:
    marker = '✔' if o.passed else '✗'
    print(f'{marker} {o.name:<46} measured={o.measured!r:<32} threshold={o.threshold!r}')